# Fine-tune NandiBERT on GLUE: SST-2 sentiment classification

SST-2 contains movie-review sentences labelled as negative or positive. The model predicts one of two classes.

We start from the released pretrained checkpoint. We are **not** pretraining a BERT model. The task-specific classification output is trained using labelled examples, and the encoder is fine-tuned as well.


## 1. Install and import

Run this once in a fresh environment. The NandiBERT model card specifies Transformers 5.x and `trust_remote_code=True`.

In [1]:
!pip install -U "transformers>=5.0" "datasets<4" accelerate scikit-learn

  Using cached transformers-5.23.0-py3-none-any.whl.metadata (31 kB)
  Using cached accelerate-1.16.0-py3-none-any.whl.metadata (19 kB)
  Using cached scikit_learn-1.9.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.3 kB)
  Using cached regex-2026.11.19-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.23.3-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.8 kB)
  Using cached typer-0.29.2-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.2 kB)
  Using cached pyarrow-26.0.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached pandas-3.1.1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (16 kB)
  Using cached xxhash-4.1.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (18 kB)
  Using cached torch-2.15

In [2]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments
)
from sklearn.metrics import accuracy_score, f1_score

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

CUDA: True
NVIDIA GeForce RTX 4090


## 2. Load the data

First inspect a row. Sentence classification gives one label to a whole sentence; sentence-pair tasks give one label to a pair.

In [4]:
dataset = load_dataset("nyu-mll/glue", "sst2")
print(dataset)
print(dataset["train"][0])

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

sst2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  148kB            

sst2/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})
{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}


## 3. Tokenize

The tokenizer converts text into token IDs and builds an attention mask. Padding is done later by the data collator, so each batch is padded only to its longest example.

In [5]:
MODEL_ID = "FrontiersMind/NandiBERT-150M"
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, trust_remote_code=True
)

config.json:   0%|          | 0.00/1.44k [00:00<?, ?B/s]

configuration_nandi_encoder.py:   0%|          | 0.00/6.07k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/FrontiersMind/NandiBERT-150M:
- configuration_nandi_encoder.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/509 [00:00<?, ?B/s]

tokenization_nandi_encoder.py:   0%|          | 0.00/5.76k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/FrontiersMind/NandiBERT-150M:
- tokenization_nandi_encoder.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer.json:   0%|          | 0.00/9.19M [00:00<?, ?B/s]

In [6]:
def tokenize(batch):
    return tokenizer(
        batch["sentence"],
        None,
        truncation=True,
        max_length=256
    )

tokenized = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

## 4. Load the classification model

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    trust_remote_code=True,
    dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float32,
)
print("Parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

modeling_nandi_encoder.py:   0%|          | 0.00/24.5k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/FrontiersMind/NandiBERT-150M:
- modeling_nandi_encoder.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors: reconstructing file:   0%|          |  0.00B /  448MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

[transformers] NandiForSequenceClassification LOAD REPORT from: FrontiersMind/NandiBERT-150M
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parameters: 146.673666 M


## 5. Metrics and training

Accuracy is easy to interpret. F1 is useful when classes are not evenly represented. For MRPC we report both accuracy and F1; for MNLI, accuracy is the standard GLUE metric.

In [8]:
def compute_metrics(output):
    predictions = output.predictions.argmax(axis=-1)
    return {"accuracy": accuracy_score(output.label_ids, predictions)}

In [9]:
args = TrainingArguments(
    output_dir="./results/sst2",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

/venv/main/lib/python3.12/site-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.182772,0.240332,0.938073
2,0.151092,0.284798,0.938073
3,0.115466,0.287525,0.943807


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=12630, training_loss=0.15760030633196978, metrics={'train_runtime': 2040.1333, 'train_samples_per_second': 99.036, 'train_steps_per_second': 6.191, 'total_flos': 6044844244282344.0, 'train_loss': 0.15760030633196978, 'epoch': 3.0})

## 6. Evaluate and save

For MNLI we evaluate both matched and mismatched validation sets. For the other tasks, we evaluate the validation split.

In [10]:
print(trainer.evaluate(tokenized["validation"]))

Training Loss,Validation Loss,Epoch,Accuracy
0.115466,0.287525,3,0.943807


{'eval_loss': 0.2875248193740845, 'eval_accuracy': 0.9438073394495413}


In [11]:
trainer.save_model("./results/sst2/final")
tokenizer.save_pretrained("./results/sst2/final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./results/sst2/final/tokenizer_config.json',
 './results/sst2/final/tokenizer.json')

## What to try next

- Compare learning rates such as `1e-5`, `2e-5` and `3e-5`.

I'm excited to see what the community will do with this model and I'm looking forward to seeing what you all build with it!

Until next time, happy coding!